In [ ]:
# Импорты необходимых для работы библиотек
import os
import shutil
import torch
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch.nn as nn
import torch.optim as optim

from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

# Загрузка датасета

In [ ]:
# скрипт для загрузки архива датасета с сайта по ссылке
!wget "https://zenodo.org/record/8387071/files/DeepSpaceYoloDataset.zip?download=1" -O DeepSpaceYoloDataset.zip

In [ ]:
# Разархивируем архив
!unzip DeepSpaceYoloDataset.zip -d Dataset

# Анализ Датасета

# Разделение на варианты

В этом разделе мы создадим N вариантов (групп) данных.  
Каждая группа получит приблизительно равную порцию изображений и соответствующих `.txt`-аннотаций из исходного датасета `DeepSpaceYoloDataset`.  
После выполнения здесь появятся папки:

In [ ]:
# Путь к корневой папке датасета
dataset_path    = '/content/Dataset/DeepSpaceYoloDataset/'
all_images_dir      = dataset_path + 'images'
all_labels_dir      = dataset_path + 'labels'

# Папка для вариантов
variant_splitting_path  = './Dataset/DeepSpaceYoloDataset_splited/'

In [ ]:
!ls /content/Dataset/DeepSpaceYoloDataset/images/ | wc -l

In [ ]:
# Параметр: сколько вариантов хотим получить
n_variants = 3

# Получаем и перемешиваем все файлы изображений
image_files = [f for f in os.listdir(all_images_dir) if f.lower().endswith('.jpg')]
random.seed(42)
random.shuffle(image_files)

# Делим на n_variants групп (размеры могут отличаться на 1 файл)
variant_groups = np.array_split(image_files, n_variants)

# Создаём папки для каждого варианта и копируем файлы

for idx, group in enumerate(variant_groups, start=1):
    var_name     = f"variant_{idx}"
    var_img_dir  = os.path.join(variant_splitting_path, var_name, 'images')
    var_lbl_dir  = os.path.join(variant_splitting_path, var_name, 'labels')

    # Создаём директории
    os.makedirs(var_img_dir, exist_ok=True)
    os.makedirs(var_lbl_dir, exist_ok=True)

    # Копируем файлы изображения и аннотации
    for img_file in group:
        base       = os.path.splitext(img_file)[0]
        src_img    = os.path.join(all_images_dir, img_file)
        dst_img    = os.path.join(var_img_dir, img_file)
        shutil.copy2(src_img, dst_img)

        src_lbl    = os.path.join(all_labels_dir, base + '.txt')
        dst_lbl    = os.path.join(var_lbl_dir, base + '.txt')
        if os.path.exists(src_lbl):
            shutil.copy2(src_lbl, dst_lbl)


**Датасет размещен в разделе _Файлы_ Colab'а.**

### Структура датасета

<pre>
DeepSpaceYoloDataset_splited/
│
├── variant_1/
│   ├── images/
│   └── labels/
├── variant_2/
│   ├── images/
│   └── labels/
...
└── variant_N/
    ├── images/
    └── labels/

    └── ...
</pre>

- **images**: содержит изображения.
- **labels**: содержит файлы аннотаций с bounding boxes для каждого изображения.

> **Примечание:** Имена файлов в обеих папках совпадают и являются id соответствующего фото.


## Посмотрим данные глазами

### Формат боксов в YOLO

В файлах аннотаций для YOLO каждая строка описывает один bounding box в формате:

<class> <x_center> <y_center> <width> <height>


- **<class>**: целое число, обозначающее класс объекта.
- **<x_center>**: координата центра бокса по оси X, нормированная (значение от 0 до 1).
- **<y_center>**: координата центра бокса по оси Y, нормированная (значение от 0 до 1).
- **<width>**: ширина бокса, нормированная относительно ширины изображения.
- **<height>**: высота бокса, нормированная относительно высоты изображения.

_Пример строки:_  

2 0.5 0.5 0.2 0.3

Это означает, что объект класса `2` расположен в центре изображения, его ширина составляет 20% от ширины изображения, а высота – 30%.


Each RGB image has a resolution of 608 × 608 pixels and corresponds to the capture of different zones of the night sky visible in Northern Hemisphere.

In [ ]:
variant = "variant_1/" # напишите цифру своего варианта
images_dir = variant_splitting_path + variant + 'images/'
labels_dir = variant_splitting_path + variant + 'labels/'

### Выведем фото и разметку на нем

In [ ]:
# Имя файла изображения и соответствующего файла разметки
# ПОТЫКАЙТЕ РАЗНЫЕ НАЗВАНИЯ ФАЙЛОВ, ЧТОБЫ ПОСМОТРЕТЬ РАЗНЫЕ ФОТО
image_filename = images_dir + '1.jpg'
annotation_filename = labels_dir + '1.txt'

# Выведем содержимое файла разметки:
image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
selected     = random.choice(image_files)

# Полные пути к файлам
image_filename      = os.path.join(images_dir, selected)
annotation_filename = os.path.join(labels_dir, os.path.splitext(selected)[0] + '.txt')

# Выведем содержимое файла разметки
print(f"\nСодержимое разметки для {selected}:")
with open(annotation_filename, 'r') as f:
    print(f.read())

# Загрузка изображения
image = Image.open(image_filename)
img_width, img_height = image.size

# Парсинг разметки в COCO-style bbox
boxes = []
with open(annotation_filename, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cls, x_c, y_c, w, h = map(float, parts)
        x_c *= img_width;  y_c *= img_height
        w   *= img_width;  h   *= img_height
        x1   = x_c - w/2;  y1   = y_c - h/2
        boxes.append((int(x1), int(y1), int(w), int(h), int(cls)))

# Визуализация
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(image)
for x1, y1, w, h, cls in boxes:
    rect = patches.Rectangle((x1, y1), w, h, linewidth=2, edgecolor='red', facecolor='none') # наносим прямоугольник (bbox)
    ax.add_patch(rect)
    ax.text(x1, y1, f'Class {cls}', color='yellow', fontsize=12, backgroundcolor='black') # наносим метку класса
plt.title(f"Изображение: {selected}")
plt.axis('off')
plt.show()

## Посмотрим статистические данные о нашем датасете

Нам интересно узнать:
- **Сколько у нас вообще фото**
- **Сколько из них размечено**
- **Сколько фото размечены некорректно**

> **Результат:**  
> Этот датасет околоидеальный, поэтому все файлы размечены корректно, и некорректных аннотаций нет.


In [ ]:
# Получаем список файлов изображений (расширения jpg, jpeg, png)
image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg'))]
total_images = len(image_files)

annotated_count = 0
empty_annotations_count = 0
invalid_annotations = []  # для хранения файлов с некорректными аннотациями

# Проходим по каждому изображению и проверяем соответствующий файл разметки
for img_file in image_files:
    # Получаем базовое имя файла (без расширения)
    base_name = os.path.splitext(img_file)[0]
    label_file = os.path.join(labels_dir, base_name + '.txt')

    # Проверяем, существует ли файл аннотации
    if os.path.exists(label_file):
        with open(label_file, 'r') as f:
            lines = f.readlines()

        # Если файл пустой, считаем аннотацию пустой
        if len(lines) == 0:
            empty_annotations_count += 1
        else:
            valid = True
            # Проверяем каждую строку аннотации
            for line in lines:
                tokens = line.strip().split()
                # Формат: <class> <x_center> <y_center> <width> <height>
                if len(tokens) != 5:
                    valid = False
                    break
                # Проверяем, что все значения можно привести к float
                try:
                    _ = list(map(float, tokens))
                except ValueError:
                    valid = False
                    break
            if valid:
                annotated_count += 1
            else:
                invalid_annotations.append(label_file)
    else:
        # Если файла аннотации нет, считаем его пустым
        empty_annotations_count += 1

print("Общее количество изображений:", total_images)
print("Количество изображений с корректной аннотацией:", annotated_count)
print("Количество изображений без аннотаций или с пустыми файлами:", empty_annotations_count)


# Подготовка датасета к обучению




Теперь сделаем главное:
- Разделим датасет на обучающую/валидационную/тестовую выборки

In [ ]:
# Пути для сохранения разделённых данных
output_dir = variant_splitting_path + "valtrain_splited/"
train_images_dir = output_dir + "train/images/"
train_labels_dir =output_dir + "train/labels/"
val_images_dir = output_dir + "val/images/"
val_labels_dir = output_dir + "val/labels/"
test_images_dir = output_dir + "test/images/"
test_labels_dir = output_dir + "test/labels/"

In [ ]:
# Получаем список всех файлов изображений
image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg'))]
image_files.sort()

# Создаем все необходимые директории, включая поддиректории
directories = [
    train_images_dir, train_labels_dir,
    val_images_dir, val_labels_dir,
    test_images_dir, test_labels_dir
]
for directory in directories:
    os.makedirs(directory, exist_ok=True)

# Разделяем файлы:
# Сначала на train (75%) и val+test (25%)
train_files, val_test_files = train_test_split(image_files, test_size=0.25, random_state=42)
# Затем из val+test выделяем валидацию(35%)
val_files, test_files = train_test_split(val_test_files, test_size=0.65, random_state=42)

print("Общее количество изображений:", len(image_files))
print("Количество изображений для обучения:", len(train_files))
print("Количество изображений для валидации:", len(val_files))
print("Количество изображений для тестирования:", len(test_files))

def copy_files(file_list, src_images, src_labels, dst_images, dst_labels):
    for file in tqdm(file_list, desc="Копирование файлов"):
        base = os.path.splitext(file)[0]
        # Копируем изображение
        shutil.copy2(os.path.join(src_images, file), os.path.join(dst_images, file))
        # Копируем соответствующий файл аннотации, если он существует
        label_file = base + ".txt"
        src_label_path = os.path.join(src_labels, label_file)
        if os.path.exists(src_label_path):
            shutil.copy2(src_label_path, os.path.join(dst_labels, label_file))

# Копируем файлы для каждого набора
copy_files(train_files, images_dir, labels_dir, train_images_dir, train_labels_dir)
copy_files(val_files, images_dir, labels_dir, val_images_dir, val_labels_dir)
copy_files(test_files, images_dir, labels_dir, test_images_dir, test_labels_dir)

# Обучение модели

## Определение конфигов и гиперпараметров

Модель YOLOv8 из ultralytics требует конфигурационного файла с путями к разным выборкам и описанием кол-ва и названий классов. Она использует его для инициализации внутри себя объектов Dataset и Dataloader, которые уже будут использоваться в процессе обучения.

In [ ]:
# Определяем содержимое файла data.yaml в виде многострочной строки

# !!!!!!     !!!!!!!!
data_yaml_content = f"""
train: {train_images_dir}
val:   {val_images_dir}
test:  {test_images_dir}

nc: 1
names: ['object']
"""

# Записываем содержимое в файл data.yaml
with open("data.yaml", "w") as file:
    file.write(data_yaml_content.strip())

print("Файл data.yaml успешно создан!")

Теперь создадим словарь с гиперпараметрами процесса обучения модели. Затем мы передадим его модели.

In [ ]:
hyperparameters = {
'lr0': 0.01,           # начальная скорость обучения
'momentum': 0.937,     # моментум
'weight_decay': 0.0005,   # коэффициент регуляризации (L2)
'warmup_epochs': 3.0,   # количество эпох разминки
'warmup_momentum': 0.8,   # моментум в фазе разминки
'warmup_bias_lr': 0.1,  # скорость обучения для bias в фазе разминки
'box': 7.5,            # коэффициент потерь для регрессии координат боксов
'cls': 0.5,            # коэффициент потерь для классификации
'label_smoothing': 0.0  # сглаживание меток
}

## Реализация обучения

Для обучения используем реализацию YOLOv8 от ultralytics. Они предоставляют собственный фреймворк обучения, который скрывает под капотом реализацию многих процессов обучения, в нашей цели, попробовать обучить большую реальную модель сгодится.

In [ ]:
!pip install ultralytics

## Ultralytics и модели семейства YOLOv8

Ultralytics сразу предоставляет файлы всех моделей из семейства YOLOv8.  
Эти файлы содержат описание архитектуры модели (какие слои с какими параметрами следуют друг за другом) и предобученные на датасетах с общими классами (яблоко, машина, дерево) веса для всех слоев модели.

---

Мы как раз соберем нашу модель с помощью `model = YOLO("yolov8n.pt")`.  
После успешной сборки модель отобразит в выводе получившуюся архитектуру.

---

Затем запустится цикл обучения. В этом фреймворке цикл обучения реализуется через вызов метода `.train()`.  
Все необходимые параметры и гиперпараметры мы передаем как аргументы для метода `.train()`. После успешной инициализации цикла обучения принятые гиперпараметры также будут выведены.

---

Также для повышения разнообразия данных, борьбы с переобучением у огромной, сложной модели мы используем метод аугментации данных. С гиперпараметрами аугментации данных также можно поиграться, чтобы получить лучшие результаты.

---

После каждой эпохи запускается инференс валидации

---

После окончания обучения можно сразу переходить к анализу результатов.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    task='detect',
    data="data.yaml",   # Файл с описанием датасета
    epochs=31,          # Количество эпох обучения
    imgsz=608,          # Размер изображений для обучения
    batch=32,           # Размер батча
    **hyperparameters,  # используем распаковку словаря, чтобы задать гиперпараметры
    device=0 if torch.cuda.is_available() else 'cpu'  ,          # Явно указываем использовать первый GPU (если доступно)

    # Параметры аугментации изображений
    fliplr=0.5,         # Вероятность горизонтального отражения
    flipud=0.0,         # Вероятность вертикального отражения
    hsv_h=0.015,        # Изменение оттенка (Hue)
    hsv_s=0.7,          # Изменение насыщенности (Saturation)
    hsv_v=0.4,          # Изменение яркости (Value)
    degrees=0.0,        # Максимальный угол поворота
    translate=0.1,      # Диапазон сдвига изображения
    scale=0.5,          # Диапазон масштабирования
    shear=0.0,          # Диапазон сдвига (shear)
    perspective=0.0     # Перспективное преобразование
    )

# После завершения обучения можно проверить результаты
print("Результаты обучения:")
print(results)

# Просмотр результатов обучения

- **Сохранение экспериментов:**  
  Результаты всех экспериментов в этом фреймворке сохраняются в каталог `/runs/`, в котором идет разделение на решаемые задачи, в нашем случае `/detect/`, в котором уже хранятся поименованные эксперименты. По умолчанию они называются `train`(соответственно далее `train2`, `train3 ...).

- **Хранение результатов обучения:**  
  Все результаты каждого обучения хранятся в соответствующей папке. Там можно найти фото с наложенными боксами, скаляры и графики.

- **Генерация метрик:**  
  Данный фреймворк "из коробки" автоматически генерирует и сохраняет матрицу ошибок и скалярные метрики итогов обучения. Так как у нас всего один класс, матрица крайне удобна для чтения.


In [ ]:
confusion_matrix = '/content/runs/detect/train/confusion_matrix.png'
img = Image.open(confusion_matrix)
img

В фреймворки Ultralytics YOLO метрики `(results/metrics)` отражают показатели, рассчитанные на валидационной выборке `(valid)`. Соответственно строя и проверяя свои гипотезы, в основном, вы должны преследовать цель улучшения метрики `(results/metrics/mAP50)`, так как это главная метрика в задаче детекции.

In [ ]:
results_img = "/content/runs/detect/train/results.png"
img = Image.open(results_img)
img

Удобнейшим инструментом для просмотра результатов экспериментов является `TensorBoard`. В нем сбоку во вкладке `Time Series` можно выбрать эксперименты для сравнения и посмотреть все логгируемые метрики

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/runs/detect/

# Проверка модели на отложенной тестовой выборке

Также, для запуска инференса модели (в т.ч. на тестовой выборке) в этом фреймворке существует метод `model.predict()`.

Для тестирования мы возьмем лучший чекпоинт(состояние весов модели в ходе обучение), который автоматически сохраняется как `best.pt`.

Загрузим из него модель и запустим инференс на тестовых данных.


In [ ]:
test_model =  YOLO("/content/runs/detect/train/weights/best.pt")

model.predict("/content/Dataset/DeepSpaceYoloDataset_splited/test/images", save=True, project="/content/runs/predict/", name="predict1", imgsz=608, conf=0.5)